# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is. The functions below
are the ones behind the Monitoring page of the app.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed


volatility_flag_lookahead completed
sarima_forecast_block blocked
prompt_injection_block blocked
frequency_mismatch_block blocked


In [3]:
from quantlab.monitoring import (
    capability_counts,
    guardrail_counts,
    kpis,
    read_events,
    recent_errors,
    stage_latency,
)

events = read_events(settings.events_path)
print(len(events), "events")
pd.Series(kpis(events))

102 events


n_runs               8.0
n_completed          4.0
n_blocked            4.0
n_failed             0.0
n_in_progress        0.0
success_rate         0.5
blocked_rate         0.5
failed_rate          0.0
n_replays            1.0
replay_match_rate    1.0
cache_hit_rate       NaN
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds):

In [4]:
pd.DataFrame(stage_latency(events))

,stage,count,mean_ms,p95_ms
0,planning,8,0.0,0.0
1,validation,16,0.2,0.3
2,execution,4,111.7,113.5
3,narration,4,0.0,0.0


Which capabilities run most, and which guardrails fire most:

In [5]:
pd.Series(capability_counts(events), name="executions")

load_series           4
resample              4
compute_returns       4
rolling_volatility    4
flag_regime           4
Name: executions, dtype: int64

In [6]:
pd.DataFrame(guardrail_counts(events))

,code,name,severity,count
0,G12,MATERIAL_ASSUMPTION,APPROVAL,14
1,G10,LOOK_AHEAD,APPROVAL,2
2,G14,WINDOW_UNITS_CHANGED,WARN,2
3,G03,INVALID_PARAM,BLOCK,2
4,G13,PLANNER_REFUSED,BLOCK,2
5,G09,FREQUENCY_MISMATCH,BLOCK,1


The most recent failures (blocked runs and errors):

In [7]:
pd.DataFrame(recent_errors(events, limit=5))

,timestamp,run_id,event_type,error
0,2026-10-16T17:55:37.106Z,20261016-175537-35cc14,run_failed,The plan has blocking issues: G09:s6
1,2026-10-16T17:55:37.071Z,20261016-175537-5d82bf,run_failed,The plan has blocking issues: G13:plan
2,2026-10-16T17:55:37.038Z,20261016-175537-24297f,run_failed,The plan has blocking issues: G13:plan
3,2026-10-16T17:55:22.245Z,20261016-175522-d8a32b,run_failed,The plan has blocking issues: G03:plan:edit:s4.capability
4,2026-10-16T17:55:22.214Z,20261016-175522-d8a32b,run_failed,The plan has blocking issues: G03:s5:quantile
